In [0]:
# ============================================
# BRONZE LAYER: Raw Data Ingestion
# ============================================

print("🏪 RETAIL SALES LAKEHOUSE PROJECT")
print("=" * 50)
print("Layer: BRONZE (Raw Data)")
print()

# Your file path
FILE_PATH = "/Volumes/workspace/default/retail_data/Online_Retail.csv"

print(f"📂 Reading: {FILE_PATH}")

🏪 RETAIL SALES LAKEHOUSE PROJECT
Layer: BRONZE (Raw Data)

📂 Reading: /Volumes/workspace/default/retail_data/Online_Retail.csv


In [0]:
# Read CSV into Spark DataFrame
bronze_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(FILE_PATH)

print(f"✅ Data loaded!")
print(f"📊 Rows: {bronze_df.count():,}")
print(f"📊 Columns: {len(bronze_df.columns)}")

✅ Data loaded!
📊 Rows: 541,909
📊 Columns: 8


In [0]:
# Show first 5 rows
print("👀 First 5 rows:")
bronze_df.show(5, truncate=False)

👀 First 5 rows:
+---------+---------+-----------------------------------+--------+----------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate     |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------------+--------+----------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |01-12-2010 08:26|2.55     |17850     |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |01-12-2010 08:26|3.39     |17850     |United Kingdom|
|536365   |84406B   |CREAM CUPID HEARTS COAT HANGER     |8       |01-12-2010 08:26|2.75     |17850     |United Kingdom|
|536365   |84029G   |KNITTED UNION FLAG HOT WATER BOTTLE|6       |01-12-2010 08:26|3.39     |17850     |United Kingdom|
|536365   |84029E   |RED WOOLLY HOTTIE WHITE HEART.     |6       |01-12-2010 08:26|3.39     |17850     |United Kingdom|
+---------+---------+---

In [0]:
# View column names and data types
print("📋 Schema (Column Types):")
bronze_df.printSchema()

📋 Schema (Column Types):
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)



In [0]:
print("📊 DATA EXPLORATION")
print("=" * 50)

# Basic counts
total_rows = bronze_df.count()
print(f"Total rows: {total_rows:,}")

# Unique counts
unique_invoices = bronze_df.select("InvoiceNo").distinct().count()
print(f"Unique invoices: {unique_invoices:,}")

unique_products = bronze_df.select("StockCode").distinct().count()
print(f"Unique products: {unique_products:,}")

unique_customers = bronze_df.select("CustomerID").distinct().count()
print(f"Unique customers: {unique_customers:,}")

unique_countries = bronze_df.select("Country").distinct().count()
print(f"Unique countries: {unique_countries:,}")

📊 DATA EXPLORATION
Total rows: 541,909
Unique invoices: 25,900
Unique products: 4,070
Unique customers: 4,373
Unique countries: 38


In [0]:
# Find earliest and latest dates
from pyspark.sql.functions import min, max

print("📅 Date Range:")
bronze_df.agg(
    min("InvoiceDate").alias("Earliest_Date"),
    max("InvoiceDate").alias("Latest_Date")
).show()

📅 Date Range:
+----------------+----------------+
|   Earliest_Date|     Latest_Date|
+----------------+----------------+
|01-02-2011 08:23|31-10-2011 17:19|
+----------------+----------------+



In [0]:
print("🔍 DATA QUALITY CHECK")
print("=" * 50)

from pyspark.sql.functions import col, sum as spark_sum

# Check null values
null_counts = bronze_df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) 
    for c in bronze_df.columns
])

print("Null values per column:")
null_counts.show()

# Count problematic rows
neg_quantity = bronze_df.filter(col("Quantity") <= 0).count()
zero_price = bronze_df.filter(col("UnitPrice") <= 0).count()

print(f"Rows with Quantity <= 0: {neg_quantity:,}")
print(f"Rows with UnitPrice <= 0: {zero_price:,}")

🔍 DATA QUALITY CHECK
Null values per column:
+---------+---------+-----------+--------+-----------+---------+----------+-------+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|
+---------+---------+-----------+--------+-----------+---------+----------+-------+
|        0|        0|       1454|       0|          0|        0|    135080|      0|
+---------+---------+-----------+--------+-----------+---------+----------+-------+

Rows with Quantity <= 0: 10,624
Rows with UnitPrice <= 0: 2,517


In [0]:
# Quick statistical summary
print("📈 Statistical Summary:")
bronze_df.select("Quantity", "UnitPrice").describe().show()

📈 Statistical Summary:
+-------+------------------+-----------------+
|summary|          Quantity|        UnitPrice|
+-------+------------------+-----------------+
|  count|            541909|           541909|
|   mean|  9.55224954743324|4.611113626089584|
| stddev|218.08115785023307|96.75985306117889|
|    min|            -80995|        -11062.06|
|    max|             80995|          38970.0|
+-------+------------------+-----------------+



In [0]:
# Top countries by transaction count
print("🌍 Top 10 Countries:")
bronze_df.groupBy("Country") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(10)

🌍 Top 10 Countries:
+--------------+------+
|       Country| count|
+--------------+------+
|United Kingdom|495478|
|       Germany|  9495|
|        France|  8557|
|          EIRE|  8196|
|         Spain|  2533|
|   Netherlands|  2371|
|       Belgium|  2069|
|   Switzerland|  2002|
|      Portugal|  1519|
|     Australia|  1259|
+--------------+------+
only showing top 10 rows


In [0]:
print("💾 SAVING BRONZE LAYER")
print("=" * 50)

# Create database
spark.sql("CREATE DATABASE IF NOT EXISTS retail_lakehouse")
print("✅ Database ready")

# Save as Delta table
bronze_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail_lakehouse.bronze_online_retail")

print("✅ Bronze table saved!")
print("📁 Table: retail_lakehouse.bronze_online_retail")

💾 SAVING BRONZE LAYER
✅ Database ready
✅ Bronze table saved!
📁 Table: retail_lakehouse.bronze_online_retail


In [0]:
print("🎯 VERIFICATION")
print("=" * 50)

# Read back the table
verify_df = spark.table("retail_lakehouse.bronze_online_retail")

# Compare counts
original = bronze_df.count()
saved = verify_df.count()

print(f"Original rows: {original:,}")
print(f"Saved table rows: {saved:,}")

if original == saved:
    print("✅ Row counts match perfectly!")
else:
    print(f"⚠️ Difference: {abs(original - saved)} rows")

# List all tables
print("\n📋 Tables in retail_lakehouse:")
spark.sql("SHOW TABLES IN retail_lakehouse").show()

🎯 VERIFICATION
Original rows: 541,909
Saved table rows: 541,909
✅ Row counts match perfectly!

📋 Tables in retail_lakehouse:
+----------------+--------------------+-----------+
|        database|           tableName|isTemporary|
+----------------+--------------------+-----------+
|retail_lakehouse|bronze_online_retail|      false|
|                |         bronze_temp|       true|
+----------------+--------------------+-----------+



In [0]:
print("=" * 50)
print("🎉 BRONZE LAYER - COMPLETE!")
print("=" * 50)
print()
print("✅ What we accomplished:")
print("   • Loaded raw CSV data")
print("   • Explored data structure")
print("   • Checked data quality")
print("   • Saved as Delta table")
print()
print("📊 Data Summary:")
print(f"   • {bronze_df.count():,} rows")
print(f"   • {len(bronze_df.columns)} columns")
print(f"   • {unique_countries} countries")
print(f"   • {unique_products:,} products")
print()
print("📁 Next Step:")
print("   Open notebook: 02_data_cleaning_silver")
print("=" * 50)

🎉 BRONZE LAYER - COMPLETE!

✅ What we accomplished:
   • Loaded raw CSV data
   • Explored data structure
   • Checked data quality
   • Saved as Delta table

📊 Data Summary:
   • 541,909 rows
   • 8 columns
   • 38 countries
   • 4,070 products

📁 Next Step:
   Open notebook: 02_data_cleaning_silver
